### Packages

In [5]:
import os
import math

### Functions

In [6]:
# import videos

# def list_mp4_files(directory):
#     # List to store mp4 file names
#     mp4_files = []

#     # Iterate through files in the given directory
#     for filename in os.listdir(directory):
#         # Check if file is an mp4
#         if filename.endswith('.mp4'):
#             mp4_files.append(filename)
    
#     return mp4_files

def list_mp4_files(directory):
    mp4_files = []
    for mouse_id in os.listdir(directory):
        mouse_path = os.path.join(directory, mouse_id)

        for day in os.listdir(mouse_path):
            day_path = os.path.join(mouse_path, day)

            for condition in os.listdir(day_path):
                condition_path = os.path.join(day_path, condition)

                target_file = f"{mouse_id}_pov_tunnel.mp4"
                full_path = os.path.join(condition_path, target_file)
                mp4_files.append(full_path)
    return mp4_files

def edit_video_bash(dir, output, mp4_files, brightness=0.0, contrast=1.0): #defaults brightness 0.0 default contrast 1.0
    # Create a bash script file
    script_path = os.path.join(dir, 'edit_videos.sh')
    
    with open(script_path, 'w') as script_file:
        # Write the bash script header
        script_file.write("#!/bin/bash\n\n")

        # Loop through each mp4 file and generate ffmpeg commands
        for video in mp4_files:
            filename = os.path.splitext(video)[0]
            input_path = os.path.join(dir, video)
            output_path = os.path.join(output, f"{filename}_edited.mp4")
            
            # Generate ffmpeg command to adjust brightness and contrast
            ffmpeg_command = f"ffmpeg -i \"{input_path}\" -vf eq=brightness={brightness}:contrast={contrast} \"{output_path}\"\n"

            # Generate ffmpeg command to convert file type
            # ffmpeg_command = f"ffmpeg -y -i \"{input_path}\" -c:v libx264 -pix_fmt yuv420p -preset superfast -crf 23 \"{output_path}\"\n"
            
            # Write the command to the bash script
            script_file.write(ffmpeg_command)

    # Make the bash script executable
    os.chmod(script_path, 0o755)
    print(f"Bash script created: {script_path}")


def edit_video_batch(dir, output, mp4_files, brightness=0.0, contrast=1.0): #defaults brightness 0.0 default contrast 1.0
    # Create a bash script file
    script_path = os.path.join(dir, 'edit_videos.')
    
    with open(script_path, 'w') as script_file:
        # Write the bash script header
        script_file.write("@echo off\n\n")

        # Loop through each mp4 file and generate ffmpeg commands
        for video in mp4_files:
            filename = os.path.splitext(video)[0]
            input_path = os.path.join(dir, video)
            output_path = os.path.join(output, f"{filename}_edited.mp4")
            
            # Generate ffmpeg command to adjust brightness and contrast
            ffmpeg_command = f"ffmpeg -i \"{input_path}\" -vf eq=brightness={brightness}:contrast={contrast} \"{output_path}\"\n"

            # Generate ffmpeg command to convert file type
            # ffmpeg_command = f"ffmpeg -y -i \"{input_path}\" -c:v libx264 -pix_fmt yuv420p -preset superfast -crf 23 \"{output_path}\"\n"
            
            # Write the command to the bash script
            script_file.write(ffmpeg_command)

    # Make the bash script executable
    os.chmod(script_path, 0o755)
    print(f"Batch script created: {script_path}")

def create_inference_bash(directory, mp4_files, model_path, batch_size=4):
    # Calculate the number of scripts needed
    num_scripts = math.ceil(len(mp4_files) / batch_size)

    # Write the bash scripts
    for script_index in range(num_scripts):
        # Create a bash script file 
        script_filename = f"{str(script_index+1).zfill(2)}_inference.sh" #name scripts
        script_path = os.path.join(directory, script_filename)

        with open(script_path, 'w') as script_file:
            # Write the bash script header (LINUX)
            script_file.write("#!/bin/bash\n\n")
            # Write the batch script header (WINDOWS)

            # Write sleap-track commands for a batch of videos
            start_index = script_index * batch_size
            end_index = min(start_index + batch_size, len(mp4_files))

            for i in range(start_index, end_index):
                video_path = os.path.join(directory, mp4_files[i])
                sleap_command = (
                    f"sleap-track \"{video_path}\" "
                    f"-m \"{model_path}\""
                    f"-o \"{video_path}.predictions.slp\n"

                    # Convert .slp files to .h5 files for analysis
                    f"sleap-convert \"{video_path}.predictions.slp\" "
                    f"--format analysis \n"
                )
                # Write the command to the bash script
                script_file.write(sleap_command)

        # Make the bash script executable
        os.chmod(script_path, 0o755)

    print(f"Created {num_scripts} bash scripts in {directory}")


def create_inference_batch(directory, mp4_files, model_path, batch_size=4):
    # Calculate the number of scripts needed
    num_scripts = math.ceil(len(mp4_files) / batch_size)

    # Write the batch scripts
    for script_index in range(num_scripts):
        # Create a batch script file 
        script_filename = f"{str(script_index+1).zfill(2)}_inference.bat" #name scripts

        script_path = os.path.join(directory, script_filename)

        with open(script_path, 'w') as script_file:

            # Write the batch script header (WINDOWS)
            script_file.write("@echo off\n\n")


            # Write sleap-track commands for a batch of videos
            start_index = script_index * batch_size
            end_index = min(start_index + batch_size, len(mp4_files))

            for i in range(start_index, end_index):
                video_path = os.path.join(directory, mp4_files[i])
                sleap_command = (
                    f"sleap-track \"{video_path}\" "
                    f"-m \"{model_path}\" "
                    f"-o \"{video_path}.predictions.slp\n"

                    # Convert .slp files to .h5 files for analysis
                    f"sleap-convert \"{video_path}.predictions.slp\" "
                    f"--format analysis \n"
                )
                # Write the command to the bash script
                script_file.write(sleap_command)

        # Make the bash script executable
        os.chmod(script_path, 0o755)

    print(f"Created {num_scripts} bash scripts in {directory}")

### Create bash script to edit video (brightness, contrast, file encoding)

In [ ]:
# make bash script to edit brightness and contrast using ffmpeg

# Change the brightness and contrast here
brightness_value = 0.5
contrast_value = 2.0   

# Linux dir format
directory_path_linux = "/oak/esay/cd_project/behavior/videos/day1"
output_directory_linux = directory_path_linux

# Windows dir format
directory_path_win = "G://My Drive/can_project/behavior/videos/day1"
output_directory_win = directory_path_win

mp4_files = list_mp4_files(directory_path_win)
edit_video_bash(directory_path_linux, output_directory_linux, mp4_files, brightness_value, contrast_value)
edit_video_batch(directory_path_win, output_directory_win, mp4_files, brightness_value, contrast_value)

Bash script created: G://My Drive/can_project/behavior/videos/day1\edit_videos.sh


### Create bash script to run inference on videos (Linux/Sherlock)
batched, run using tmux

In [ ]:
directory_path = '/oak/esay/cd_project/behavior/videos/day1'
# mp4_files = ['video1.mp4', 'video2.mp4', 'video3.mp4', 'video4.mp4', 'video5.mp4']  # Example list of videos
model_path = '/oak/esay/cd_project/behavior/v3_model/models/241210_143147.single_instance.n=1794'  # Replace with the actual path to your model

# Create bash scripts, each with 4 videos
mp4_files = list_mp4_files(directory_path)
create_inference_bash(directory_path, mp4_files, model_path, batch_size=4)


Created 2 bash scripts in /Users/ellasay/Google Drive/My Drive/can_project/behavior/videos/day1


### Write batch script (Windows)

In [3]:
directory_path = 'C:/Users/esay/data/social_interaction/SLEAP/SLEAP_raw/social-8564-2'
# mp4_files = ['video1.mp4', 'video2.mp4', 'video3.mp4', 'video4.mp4', 'video5.mp4']  # Example list of videos
model_path = 'C:/Users/esay/data/social_interaction/SLEAP/models/250127_164147.single_instance.n=2844'  # Replace with the actual path to your model

# Create bash scripts, each with 4 videos
mp4_files = list_mp4_files(directory_path)
create_inference_batch(directory_path, mp4_files, model_path, batch_size=4)


Created 15 bash scripts in C:/Users/esay/data/social_interaction/SLEAP/SLEAP_raw/social-8564-2


In [4]:
mp4_files

['C:/Users/esay/data/social_interaction/SLEAP/SLEAP_raw/social-8564-2\\day1\\empty_tunnel\\social-8564-2_pov_face.mp4\\day1_pov_tunnel.mp4',
 'C:/Users/esay/data/social_interaction/SLEAP/SLEAP_raw/social-8564-2\\day1\\empty_tunnel\\social-8564-2_pov_face.txt\\day1_pov_tunnel.mp4',
 'C:/Users/esay/data/social_interaction/SLEAP/SLEAP_raw/social-8564-2\\day1\\empty_tunnel\\social-8564-2_pov_tunnel.mp4\\day1_pov_tunnel.mp4',
 'C:/Users/esay/data/social_interaction/SLEAP/SLEAP_raw/social-8564-2\\day1\\empty_tunnel\\social-8564-2_pov_tunnel.txt\\day1_pov_tunnel.mp4',
 'C:/Users/esay/data/social_interaction/SLEAP/SLEAP_raw/social-8564-2\\day1\\unrestricted_fam\\social-8564-2_pov_face.mp4\\day1_pov_tunnel.mp4',
 'C:/Users/esay/data/social_interaction/SLEAP/SLEAP_raw/social-8564-2\\day1\\unrestricted_fam\\social-8564-2_pov_face.txt\\day1_pov_tunnel.mp4',
 'C:/Users/esay/data/social_interaction/SLEAP/SLEAP_raw/social-8564-2\\day1\\unrestricted_fam\\social-8564-2_pov_tunnel.mp4\\day1_pov_tunnel.m

In [8]:
directory_path = 'C:/Users/esay/data/social_interaction/SLEAP/SLEAP_raw/shank3_new/'
mp4_files = list_mp4_files(directory_path)


In [9]:
model_path = 'C:/Users/esay/data/social_interaction/SLEAP/models/250127_164147.single_instance.n=2844'  # Replace with the actual path to your model

create_inference_batch(directory_path, mp4_files, model_path, batch_size=4)


Created 4 bash scripts in C:/Users/esay/data/social_interaction/SLEAP/SLEAP_raw/shank3_new/
